# Salih's notebook: evaluation and baselines

CENG313 Assignment 1, BYU Locating Bacterial Flagellar Motors 2025.

My part: the F2 metric, the validation split and the baseline models (B0, B1, B2).
The code is in `src/` (`utils.py`, `models.py`). This notebook runs it and shows the results.

**Before running:** add two secrets in Colab (key icon on the left):
- `KAGGLE_API_TOKEN`: Kaggle, Settings, API, Create New Token
- `GITHUB_TOKEN`: GitHub classic token with `repo` scope (our repo is private)

## 0. Setup

In [2]:
# Kaggle token comes from Colab secrets, it is not written in the notebook
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
os.chdir("/content")
!pip -q install -U kaggle

# For the metric, the split, B0 and B1 we only need the labels file
!mkdir -p data
!kaggle competitions download -c byu-locating-bacterial-flagellar-motors-2025 -f train_labels.csv -p data
!cd data && (unzip -o -q train_labels.csv.zip 2>/dev/null || true)
!ls data

100% 37.3k/37.3k [00:00<00:00, 31.8MB/s]

train_labels.csv


Download the downsampled tomograms from our group's Kaggle dataset
(648 tomograms, 8 times smaller on each axis, block mean). B2 uses these files.

In [3]:
!kaggle datasets download -d yasunl/group10-byu-downsampled-8x -p data --unzip
!ls data
!ls data/downsampled | wc -l      # should be 648
!du -sh data/downsampled          # total size, about 0.35 GB

Dataset URL: https://www.kaggle.com/datasets/yasunl/group10-byu-downsampled-8x
License(s): unknown
100% 348M/348M [00:03<00:00, 99.2MB/s]

downsampled  examples  labels_downsampled.csv  train_labels.csv
648
331M	data/downsampled


In [4]:
# Clone our repo (my branch) so we can import src/
import sys

token = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/yasunl/GROUP10_CENG313.git"
REPO_DIR = "/content/GROUP10_CENG313"
BRANCH = "salih-baseline"

if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
!ls {REPO_DIR}/src {REPO_DIR}/tests

/content/GROUP10_CENG313/src:
data.py  __init__.py  models.py  plots.py  utils.py

/content/GROUP10_CENG313/tests:
test_utils.py


## 1. Metric check

`utils.score` works like the competition metric. A prediction is a TP if it is closer
than 1000 Å to a true motor. If the prediction is too far, or we say "no motor" when there
is a motor, it is a FN. If we predict a motor in a tomogram without a motor, it is a FP.
Then we compute F2.

`tests/test_utils.py` checks `score` on a small example. I also calculated this example by hand:
TP = 2, FP = 1, FN = 2, so precision = 2/3, recall = 1/2 and F2 = 10/19 ≈ 0.526.
The test gives the same numbers.

In [5]:
!cd {REPO_DIR} && python tests/test_utils.py

0.5263157894736842 {'tp': 2, 'fp': 1, 'fn': 2}
OK


## 2. Data overview

In [6]:
from src import data, utils, models

labels = data.load_labels("data/train_labels.csv")   # one row per motor, short column names
tomos = data.tomogram_table(labels)                   # one row per tomogram

print("label rows (motors):", len(labels))
print("tomograms          :", len(tomos))
print("with a motor       :", tomos.has_motor.sum(), f"({tomos.has_motor.mean():.1%})")
print("motors per tomogram:\n", tomos.n_motors.value_counts().sort_index())
print("voxel spacing (Å)  :\n", tomos.voxel_spacing.describe())

label rows (motors): 737
tomograms          : 648
with a motor       : 362 (55.9%)
motors per tomogram:
 n_motors
0     286
1     313
2      30
3       6
4       9
6       3
10      1
Name: count, dtype: int64
voxel spacing (Å)  :
 count    648.000000
mean      15.066204
std        2.152598
min        6.500000
25%       13.100000
50%       15.600000
75%       16.100000
max       19.700000
Name: voxel_spacing, dtype: float64


**Observations**
- 648 tomograms but 737 label rows, because a tomogram with more than one motor has one row for each motor.
- 362 tomograms (55.9%) have a motor and 286 have none, so the two groups are close in size.
- Most tomograms with a motor have only one, but 49 have 2 to 10 motors.
- Voxel spacing is between 6.5 and 19.7 Å, so 1000 Å is between about 51 and 154 voxels.
  We must change distances to Å before we compare them with 1000.
- The `Number of motors` column gives 455 motors, but there are only 451 motor coordinates (4 are missing).

## 3. Validation split

We split by tomogram, so one tomogram is never in both train and validation (no leakage).
The split is stratified by `has_motor`, so both parts have the same rate of motors.
The seed is fixed, so we get the same split every time.

In [7]:
train_ids, val_ids = utils.split_tomograms(tomos)   # val_size=0.2, seed=42

print("train:", len(train_ids), " val:", len(val_ids))
print("shared tomograms:", len(set(train_ids) & set(val_ids)))   # must be 0

train_rate = tomos[tomos.tomo_id.isin(train_ids)].has_motor.mean()
val_rate = tomos[tomos.tomo_id.isin(val_ids)].has_motor.mean()
print(f"positive rate  train: {train_rate:.1%}   val: {val_rate:.1%}")

train: 518  val: 130
shared tomograms: 0
positive rate  train: 55.8%   val: 56.2%


In [8]:
# with the same seed we get the same split
a, _ = utils.split_tomograms(tomos)
b, _ = utils.split_tomograms(tomos)
c, _ = utils.split_tomograms(tomos, seed=7)
print("same seed, same split:     ", a == b)   # True
print("different seed, same split:", a == c)   # False

same seed, same split:      True
different seed, same split: False


I save the validation ids in `splits/val_ids.csv`. This way every model (B0, B1, B2 and
our Assignment 2 model) is tested on the same 130 tomograms, even if the code or a library changes later.

In [9]:
import pandas as pd

os.makedirs(f"{REPO_DIR}/splits", exist_ok=True)
pd.Series(sorted(val_ids), name="tomo_id").to_csv(f"{REPO_DIR}/splits/val_ids.csv", index=False)
!head -5 {REPO_DIR}/splits/val_ids.csv
!wc -l {REPO_DIR}/splits/val_ids.csv

tomo_id
tomo_02862f
tomo_03437b
tomo_066095
tomo_06e11e
131 /content/GROUP10_CENG313/splits/val_ids.csv


## 4. Baselines B0 and B1

B0 always says "no motor". B1 always puts a motor at the average position of the train motors
(position divided by the tomogram size, then multiplied by the size of the new tomogram).
Both use the 518 train tomograms and are scored on the 130 validation tomograms
(ids from `splits/val_ids.csv`).

In [10]:
# use the saved validation ids
val_ids = pd.read_csv(f"{REPO_DIR}/splits/val_ids.csv")["tomo_id"].tolist()
train_ids = [t for t in tomos.tomo_id if t not in set(val_ids)]

train_labels = labels[labels.tomo_id.isin(train_ids)]
val_labels = labels[labels.tomo_id.isin(val_ids)]
val_tomos = tomos[tomos.tomo_id.isin(val_ids)]

results = {}
for name, model in [("B0", models.NoMotorBaseline()), ("B1", models.MeanPositionBaseline())]:
    preds = model.fit(train_labels).predict(val_tomos)
    f2, counts = utils.score(val_labels, preds, return_counts=True)
    p = counts["tp"] / (counts["tp"] + counts["fp"]) if counts["tp"] + counts["fp"] else 0.0
    r = counts["tp"] / (counts["tp"] + counts["fn"]) if counts["tp"] + counts["fn"] else 0.0
    results[name] = {"precision": round(p, 4), "recall": round(r, 4), "F2": round(f2, 4), **counts}

results_df = pd.DataFrame(results).T
results_df

,precision,recall,F2,tp,fp,fn
B0,0.00,0.0000,0.0000,0.0,0.0,73.0
B1,0.05,0.0411,0.0426,3.0,57.0,70.0


**Observations (B0, B1)**
- B0 says "no motor" everywhere, so it misses all 73 motors and F2 = 0. This is the lowest possible score.
- B1 puts a motor in every tomogram. All 57 tomograms without a motor become FP, and only 3 of the
  73 motors are closer than 1000 Å to the average position (F2 = 0.043).
  Motors are in many different places, so one fixed position does not work.
- So a useful model must look at the image. This is the idea of B2.

## 5. B2: patch classifier + sliding window

B2 uses the downsampled tomograms. A classifier looks at small 16x16 2D patches and says
"motor" or "no motor". Then we slide a window over the whole tomogram and take the place with
the highest motor probability. We try two classifiers: Logistic Regression (LR) and Random Forest (RF).
The code is in `src/models.py` (`PatchClassifierBaseline`).

**5.1 Downsampled data.** One file has the small volume and some info (original size,
voxel spacing, factor 8). To go back to original voxels: small coordinate x 8 + 4 (center of the 8x8x8 block).

In [11]:
ds = pd.read_csv("data/labels_downsampled.csv")
print("columns:", ds.columns.tolist())
display(ds.head())

tid = "tomo_003acc"   # one example (this tomogram has no motor)
vol, meta = data.load_downsampled(f"data/downsampled/{tid}.npz")
print("shape:", vol.shape, " dtype:", vol.dtype, " min/max:", vol.min(), vol.max())
print("meta:", meta)
print(ds[ds.tomo_id == tid])

columns: ['row_id', 'tomo_id', 'Motor axis 0', 'Motor axis 1', 'Motor axis 2', 'Array shape (axis 0)', 'Array shape (axis 1)', 'Array shape (axis 2)', 'Voxel spacing', 'Number of motors', 'z_s', 'y_s', 'x_s']


,row_id,tomo_id,Motor axis 0,Motor axis 1,Motor axis 2,Array shape (axis 0),Array shape (axis 1),Array shape (axis 2),Voxel spacing,Number of motors,z_s,y_s,x_s
0,0,tomo_003acc,-1.0,-1.0,-1.0,500,1912,1847,6.5,0,-1.000,-1.000,-1.000
1,1,tomo_00e047,169.0,546.0,603.0,300,959,928,15.6,1,21.125,68.250,75.375
2,2,tomo_00e463,235.0,403.0,137.0,500,924,956,19.7,6,29.375,50.375,17.125
3,3,tomo_00e463,243.0,363.0,153.0,500,924,956,19.7,6,30.375,45.375,19.125
4,4,tomo_00e463,222.0,379.0,144.0,500,924,956,19.7,6,27.750,47.375,18.000


shape: (62, 239, 231)  dtype: uint8  min/max: 10 190
meta: {'orig_z': 500, 'orig_y': 1912, 'orig_x': 1847, 'voxel_spacing': 6.5, 'factor': 8}
   row_id      tomo_id  Motor axis 0  Motor axis 1  Motor axis 2  \
0       0  tomo_003acc          -1.0          -1.0          -1.0   

   Array shape (axis 0)  Array shape (axis 1)  Array shape (axis 2)  \
0                   500                  1912                  1847   

   Voxel spacing  Number of motors  z_s  y_s  x_s  
0            6.5                 0 -1.0 -1.0 -1.0  


**5.2 Training.** Positive patches come from the motors in the train tomograms. We also take the
slice above and below and move the patch 1 pixel, to get more motor examples.
Negative patches are random places more than 2000 Å away from all motors.
We use `class_weight="balanced"` because we have about 3 to 4 times more negatives than positives.

**Why v2?** Our first version (v1) had only random negatives, it searched the whole volume and it
did not smooth the probabilities. On validation it was very bad: LR F2 = 0.000 (0 TP) and RF F2 = 0.017 (1 TP).
The patch classifier itself was good (patch AUC 0.93 for LR and 0.96 for RF), but the search found
the wrong places: 63% (LR) and 47% (RF) of the predictions were near the border, and the median
distance to the motor was 7150 Å (LR) and 5674 Å (RF). One tomogram has about 100,000 windows,
so some strange background places (border, bright spots) get a higher score than the motor.
So in v2 we added:
- hard negatives: we run the first model on the train tomograms, take the wrong "motor" places
  and train again with them as negatives (only train tomograms, not validation),
- margin: we do not search the outer 10% of each axis,
- smoothing: we average each window with its neighbours (3x3x3).

In [12]:
# Block 5.2: train B2 with LR and RF (v2 settings)
import glob, importlib, time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
importlib.reload(models)

VOL_DIR = "/content/data/downsampled"   # downloaded in section 0
print(len(glob.glob(f"{VOL_DIR}/*.npz")), "downsampled volumes")

# v2: 1 round of hard negatives (10 per tomogram), 10% border margin, 3x3x3 smoothing.
# refine is off here, we test it in 5.5
B2_SETTINGS = dict(hard_rounds=1, hard_per_tomo=10, margin=0.1, smooth=3, refine=False)

b2_models = {
    "B2-LR": models.PatchClassifierBaseline(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
        VOL_DIR, **B2_SETTINGS),
    "B2-RF": models.PatchClassifierBaseline(
        RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=0),
        VOL_DIR, **B2_SETTINGS),
}
for name, m in b2_models.items():
    t = time.time()
    m.fit(train_labels)
    print(f"{name}: {m.n_pos_} positive / {m.n_neg_} negative patches "
          f"({m.n_hard_} hard negatives), {time.time() - t:.0f}s")

648 downsampled volumes
B2-LR: 5311 positive / 20713 negative patches (5173 hard negatives), 61s
B2-RF: 5311 positive / 20467 negative patches (4927 hard negatives), 517s


**5.3 Validation.** We run the sliding window on the 130 validation tomograms one time,
then we try some thresholds. If the probability is lower than the threshold, we say "no motor".

In [13]:
# Block 5.3: predict the validation tomograms, then try some thresholds
b2_preds = {}
for name, m in b2_models.items():
    t = time.time()
    b2_preds[name] = m.predict(val_tomos)
    print(f"{name}: predicted {len(val_tomos)} tomograms in {time.time() - t:.0f}s")

rows = []
for name, p in b2_preds.items():
    for th in [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
        f2, c = utils.score(val_labels, models.with_threshold(p, th), return_counts=True)
        prec = c["tp"] / (c["tp"] + c["fp"]) if c["tp"] + c["fp"] else 0.0
        rec = c["tp"] / (c["tp"] + c["fn"]) if c["tp"] + c["fn"] else 0.0
        rows.append({"model": name, "threshold": th, "F2": round(f2, 4), **c,
                     "precision": round(prec, 3), "recall": round(rec, 3)})
b2_results = pd.DataFrame(rows)
b2_results

B2-LR: predicted 130 tomograms in 12s
B2-RF: predicted 130 tomograms in 77s


,model,threshold,F2,tp,fp,fn,precision,recall
0,B2-LR,0.00,0.0426,3,57,70,0.050,0.041
1,B2-LR,0.30,0.0426,3,57,70,0.050,0.041
2,B2-LR,0.50,0.0426,3,57,70,0.050,0.041
3,B2-LR,0.70,0.0000,0,31,73,0.000,0.000
4,B2-LR,0.80,0.0000,0,12,73,0.000,0.000
5,B2-LR,0.90,0.0000,0,2,73,0.000,0.000
6,B2-LR,0.95,0.0000,0,0,73,0.000,0.000
7,B2-RF,0.00,0.2060,15,57,58,0.208,0.205
8,B2-RF,0.30,0.1054,7,33,66,0.175,0.096
9,B2-RF,0.50,0.0000,0,0,73,0.000,0.000


**5.4 Check the model.**
- Patch AUC: can the classifier separate motor patches from random background patches?
- Distance: how far are the predictions from the true motor?
- Border: how many predictions are near the border?

The second cell checks how many train motors are in the outer 10% that we do not search.

In [14]:
# Block 5.4: patch AUC, distance to the motor, predictions near the border
import numpy as np
from sklearn.metrics import roc_auc_score

Xv, yv = models.extract_training_patches(val_labels, VOL_DIR, seed=1)
motors = val_labels[val_labels["z"] >= 0]
for name, m in b2_models.items():
    auc = roc_auc_score(yv, m.classifier.predict_proba(Xv)[:, 1])
    p = b2_preds[name].merge(val_tomos[["tomo_id", "shape_z", "shape_y", "shape_x"]], on="tomo_id")
    dist = []
    for tomo_id, g in motors.groupby("tomo_id"):
        r = p.loc[p.tomo_id == tomo_id, ["z", "y", "x"]].to_numpy(dtype=float)
        dv = np.sqrt(((g[["z", "y", "x"]].to_numpy(dtype=float) - r) ** 2).sum(axis=1)).min()
        dist.append(dv * g["voxel_spacing"].iloc[0])
    dist = np.array(dist)
    rel = np.stack([p[a] / p[f"shape_{a}"] for a in ("z", "y", "x")], axis=1)
    near_edge = ((rel < 0.1) | (rel > 0.9)).any(axis=1).mean()
    print(f"{name}: patch ROC AUC on val = {auc:.3f} | best-window prob median = {p.prob.median():.3f}")
    print(f"   distance to true motor (A): median {np.median(dist):.0f}, "
          f"<1000: {(dist < 1000).sum()}, <2000: {(dist < 2000).sum()}, <5000: {(dist < 5000).sum()} of {len(dist)}")
    print(f"   predictions near the volume border (outer 10%): {near_edge:.0%}")

B2-LR: patch ROC AUC on val = 0.794 | best-window prob median = 0.679
   distance to true motor (A): median 6879, <1000: 3, <2000: 5, <5000: 21 of 73
   predictions near the volume border (outer 10%): 11%
B2-RF: patch ROC AUC on val = 0.957 | best-window prob median = 0.306
   distance to true motor (A): median 1898, <1000: 15, <2000: 37, <5000: 51 of 73
   predictions near the volume border (outer 10%): 8%


In [15]:
# how many train motors are in the outer 10% (we do not search there)
m_ = train_labels[train_labels["z"] >= 0]
rel = np.stack([m_[a] / m_[f"shape_{a}"] for a in ("z", "y", "x")], axis=1)
print(f"training motors inside the outer 10% border: {((rel < 0.1) | (rel > 0.9)).any(axis=1).mean():.1%}")

training motors inside the outer 10% border: 11.7%


**5.5 Refine.** RF has 37 predictions closer than 2000 Å but only 15 closer than 1000 Å.
The search only checks every 4th pixel and every 2nd slice (about 500 Å).
So around the best window we check every pixel and every slice.
We use the same trained models, there is no new training.

In [16]:
# Block 5.5: refine the best position (same trained models)
importlib.reload(models)

def motor_distances(preds):
    """Distance in Angstrom from each true motor to the prediction for its tomogram."""
    m_ = val_labels[val_labels["z"] >= 0].merge(preds, on="tomo_id", suffixes=("", "_pred"))
    diff = m_[["z", "y", "x"]].to_numpy(dtype=float) - m_[["z_pred", "y_pred", "x_pred"]].to_numpy(dtype=float)
    m_["dist"] = np.sqrt((diff ** 2).sum(axis=1)) * m_["voxel_spacing"]
    return m_.groupby("tomo_id")["dist"].min().to_numpy()   # closest motor for each tomogram

b2_preds_refined = {}
for name, m in b2_models.items():
    t = time.time()
    refined = models.PatchClassifierBaseline(m.classifier, VOL_DIR, **dict(B2_SETTINGS, refine=True))
    b2_preds_refined[name] = refined.predict(val_tomos)
    print(f"{name} (refinement took {time.time() - t:.0f}s)")
    for label, p in (("coarse ", b2_preds[name]), ("refined", b2_preds_refined[name])):
        f2, c = utils.score(val_labels, p, return_counts=True)
        dist = motor_distances(p)
        print(f"   {label}: F2 {f2:.4f} {c} | median distance {np.median(dist):.0f} A, "
              f"<1000: {(dist < 1000).sum()}, <2000: {(dist < 2000).sum()} of {len(dist)}")

B2-LR (refinement took 12s)
   coarse : F2 0.0426 {'tp': 3, 'fp': 57, 'fn': 70} | median distance 6879 A, <1000: 3, <2000: 5 of 73
   refined: F2 0.0285 {'tp': 2, 'fp': 57, 'fn': 71} | median distance 6920 A, <1000: 2, <2000: 6 of 73
B2-RF (refinement took 83s)
   coarse : F2 0.2060 {'tp': 15, 'fp': 57, 'fn': 58} | median distance 1898 A, <1000: 15, <2000: 37 of 73
   refined: F2 0.2582 {'tp': 19, 'fp': 57, 'fn': 54} | median distance 2056 A, <1000: 19, <2000: 36 of 73


**5.6 Threshold for RF.** We try more thresholds between 0 and 0.31 (taken from the RF probabilities).

In [17]:
# Block 5.6: more thresholds for B2-RF
p = b2_preds_refined["B2-RF"]
print(p["prob"].describe().round(3))
rows = []
for th in sorted(set([0.0] + list(np.quantile(p["prob"], np.linspace(0.05, 0.6, 12)).round(3)))):
    f2, c = utils.score(val_labels, models.with_threshold(p, th), return_counts=True)
    rows.append({"threshold": th, "F2": round(f2, 4), **c,
                 "predicted_no_motor": int((p["prob"] < th).sum())})
pd.DataFrame(rows)

count    130.000
mean       0.308
std        0.052
min        0.046
25%        0.282
50%        0.306
75%        0.346
max        0.434
Name: prob, dtype: float64


,threshold,F2,tp,fp,fn,predicted_no_motor
0,0.000,0.2582,19,57,54,0
1,0.232,0.2329,17,56,56,6
2,0.250,0.1944,14,54,59,14
3,0.266,0.1826,13,51,60,20
4,0.275,0.1705,12,48,61,26
5,0.282,0.1714,12,46,61,34
6,0.286,0.1719,12,45,61,41
7,0.287,0.1734,12,42,61,44
8,0.296,0.1627,11,35,62,52
9,0.303,0.1497,10,32,63,59


**B2 results (validation, 130 tomograms)**

| model | F2 | TP | FP | FN | precision | recall |
|---|---|---|---|---|---|---|
| B0 no motor | 0.0000 | 0 | 0 | 73 | – | 0.000 |
| B1 mean position | 0.0426 | 3 | 57 | 70 | 0.050 | 0.041 |
| B2-LR v2 + refine | 0.0285 | 2 | 57 | 71 | 0.034 | 0.027 |
| **B2-RF v2 + refine** | **0.2582** | 19 | 57 | 54 | 0.250 | 0.260 |

**Observations (B2)**
- The v2 changes raised B2-RF from F2 0.017 to 0.206, and refine raised it to 0.258. This is about 6 times better than B1.
- 37 of 73 RF predictions are closer than 2000 Å, but only 19 are closer than 1000 Å.
  The model often finds the right area, but not the exact place.
- LR cannot separate the hard negatives (patch AUC 0.79, RF 0.96), so RF is our main B2 model.
- The best threshold is 0, so B2 always predicts a motor. Most probabilities are between 0.23 and 0.43,
  and they are similar for tomograms with and without a motor. F2 cares about recall 4 times more than
  precision, so removing predictions loses more TP than it saves FP.
  B2 can find where the motor is, but it cannot say if there is a motor or not.
- We chose the v2 changes by looking at this validation set, so this score is a bit too optimistic.
  The Kaggle score will show the real result.
- 11.7% of the train motors are in the outer 10% that we do not search, so recall can be at most about 88%.

## 6. Summary

| model | validation F2 | Kaggle F2 (public / private) |
|---|---|---|
| B0 no motor | 0.000 | 0.000 / 0.000 |
| B1 mean position | 0.043 | 0.010 / 0.008 |
| B2-RF (v2 + refine) | 0.258 | 0.102 / 0.075 |

- On Kaggle, B2-RF gets 0.102 (public) and 0.075 (private). This is about 10 times better than B1, so B2 also works on new tomograms.
- For Kaggle, B2 is trained on all 648 train tomograms (`kaggle/submit_b2.py`), with the same settings and threshold 0.
- The Kaggle scores are lower than validation. B1 also drops (0.043 to 0.010) and B1 has no tuning,
  so a big part of the drop comes from the hidden test set being harder than our validation set.
  Also, our validation score for B2 is a bit optimistic (see 5.6).
- Public and private scores use different test tomograms, so they are not the same.